# 1 - Crear Spark Session

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RetailStreamingLambda")
    .config(
        "spark.jars",
        "/opt/spark/jars/spark-sql-kafka-0-10_2.12-3.5.1.jar,/opt/spark/jars/spark-token-provider-kafka-0-10_2.12-3.5.1.jar,/opt/spark/jars/kafka-clients-3.6.1.jar,/opt/spark/jars/commons-pool2-2.11.1.jar"
    )
    .getOrCreate()
)

spark.version

/usr/local/lib/python3.8/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
26/08/12 01:20:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


'3.5.1'

In [2]:
!ls /opt/spark/jars/

commons-pool2-2.11.1.jar     spark-sql-kafka-0-10_2.12-3.5.1.jar
kafka-clients-3.6.1.jar      spark-token-provider-kafka-0-10_2.12-3.5.1.jar
mysql-connector-j-8.4.0.jar


# 2 - Verificar conexión con Kafka

In [2]:
kafka_server = "kafka:9092"

print(kafka_server)

kafka:9092


# 3 - Leer orders desde Kafka

In [3]:
orders_raw = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        kafka_server
    )
    .option(
        "subscribe",
        "orders_topic"
    )
    .option(
        "startingOffsets",
        "latest"
    )
    .load()
)

orders_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



# 4 - Convertir JSON de orders

In [4]:
from pyspark.sql.types import *

orders_schema = StructType([
    StructField(
        "order_id",
        IntegerType()
    ),
    StructField(
        "order_date",
        TimestampType()
    ),
    StructField(
        "order_customer_id",
        IntegerType()
    ),
    StructField(
        "order_status",
        StringType()
    )
])

### Trasformamos

In [5]:
from pyspark.sql.functions import *


orders_stream = (
    orders_raw
    .selectExpr(
        "CAST(value AS STRING) json"
    )
    .select(
        from_json(
            col("json"),
            orders_schema
        ).alias("data")
    )
    .select("data.*")
)


orders_stream.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)



# 5 - Mostrar streaming de orders

In [6]:
query_orders = (
    orders_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "truncate",
        False
    )
    .start()
)

26/08/12 01:21:18 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-188c649d-b64b-4bd2-b46f-ab35ee5b6a36. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/12 01:21:18 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+--------+----------+-----------------+------------+
|order_id|order_date|order_customer_id|order_status|
+--------+----------+-----------------+------------+
+--------+----------+-----------------+------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+--------+-------------------+-----------------+------------+
|order_id|order_date         |order_customer_id|order_status|
+--------+-------------------+-----------------+------------+
|68915   |2026-08-12 01:21:22|4940             |COMPLETE    |
+--------+-------------------+-----------------+------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+--------+-------------------+-----------------+------------+
|order_id|order_date         |order_customer_id|order_status|
+--------+-------------------+-----------------+------------+
|68916   |2026-08-12 01:21:27|404              |PENDING     |
+--------+-------------------+-----------------+------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+--------+-------------------+-----------------+------------+
|order_id|order_date         |order_customer_id|order_status|
+--------+-------------------+-----------------+------------+
|68917   |2026-08-12 01:21:33|7724             |CLOSED      |
+--------+-------------------+-----------------+------------+



-------------------------------------------
Batch: 4
-------------------------------------------
+--------+-------------------+-----------------+------------+
|order_id|order_date         |order_customer_id|order_status|
+--------+-------------------+-----------------+------------+
|68918   |2026-08-12 01:21:40|5189             |CLOSED      |
+--------+-------------------+-----------------+------------+

-------------------------------------------
Batch: 5
-------------------------------------------
+--------+-------------------+-----------------+------------+
|order_id|order_date         |order_customer_id|order_status|
+--------+-------------------+-----------------+------------+
|68919   |2026-08-12 01:21:47|6774             |CLOSED      |
+--------+-------------------+-----------------+------------+

-------------------------------------------
Batch: 6
-------------------------------------------
+--------+-------------------+-----------------+------------+
|order_id|order_date     

In [7]:
query_orders.stop()

# 6 - Leer order_items

In [8]:
items_raw = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        kafka_server
    )
    .option(
        "subscribe",
        "order_items_topic"
    )
    .option(
        "startingOffsets",
        "latest"
    )
    .load()
)

In [9]:
items_schema = StructType([

    StructField(
        "order_item_id",
        IntegerType()
    ),

    StructField(
        "order_item_order_id",
        IntegerType()
    ),

    StructField(
        "order_item_product_id",
        IntegerType()
    ),

    StructField(
        "order_item_quantity",
        IntegerType()
    ),

    StructField(
        "order_item_subtotal",
        DoubleType()
    ),

    StructField(
        "order_item_product_price",
        DoubleType()
    )
])

In [10]:
items_stream = (
    items_raw
    .selectExpr(
        "CAST(value AS STRING) json"
    )
    .select(
        from_json(
            col("json"),
            items_schema
        ).alias("data")
    )
    .select("data.*")
)


items_stream.printSchema()

root
 |-- order_item_id: integer (nullable = true)
 |-- order_item_order_id: integer (nullable = true)
 |-- order_item_product_id: integer (nullable = true)
 |-- order_item_quantity: integer (nullable = true)
 |-- order_item_subtotal: double (nullable = true)
 |-- order_item_product_price: double (nullable = true)



# 7 - Mostrar items

In [11]:
query_items = (
    items_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "truncate",
        False
    )
    .start()
)

26/08/12 01:22:35 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-0739dbe9-1811-4678-a018-acf9f81fdd1a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/12 01:22:35 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
|order_item_id|order_item_order_id|order_item_product_id|order_item_quantity|order_item_subtotal|order_item_product_price|
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
|order_item_id|order_item_order_id|order_item_product_id|order_item_quantity|order_item_subtotal|order_item_product_price|
+-------------+-------------------+---------------------+----------

-------------------------------------------
Batch: 6
-------------------------------------------
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
|order_item_id|order_item_order_id|order_item_product_id|order_item_quantity|order_item_subtotal|order_item_product_price|
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
|172339       |68931              |1283                 |4                  |139.96             |34.99                   |
|172338       |68931              |273                  |4                  |111.96             |27.99                   |
|172340       |68931              |1246                 |3                  |29.97              |9.99                    |
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+

-----------------------------------------

-------------------------------------------
Batch: 8
-------------------------------------------
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
|order_item_id|order_item_order_id|order_item_product_id|order_item_quantity|order_item_subtotal|order_item_product_price|
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
|172343       |68933              |962                  |3                  |179.94             |59.98                   |
|172344       |68933              |226                  |2                  |1199.98            |599.99                  |
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+



In [13]:
query_items.stop()

In [14]:
from pyspark.sql.functions import *
# "La columna order_date representa el tiempo del evento. Permite que lleguen datos atrasados hasta 10 minutos."
orders_stream_watermark = (
    orders_stream
    .withWatermark(
        "order_date",
        "2 minutes"
    )
)

In [15]:
#"Para este stream, considera válidos los eventos atrasados hasta 10 minutos usando event_time."
items_stream_watermark = (
    items_stream
    .withColumn(
        "event_time",
        current_timestamp()
    )
    .withWatermark(
        "event_time",
        "2 minutes"
    )
)

# Revisa un procesamiento en medio 

In [16]:
join_condition = (
    orders_stream_watermark.order_id ==
    items_stream_watermark.order_item_order_id
)

In [17]:
sales_stream = (
    orders_stream_watermark
    .join(
        items_stream_watermark,
        join_condition,
        "inner"
    )
)

In [18]:
ventas = (
    sales_stream
    .select(
        orders_stream_watermark.order_id,
        orders_stream_watermark.order_customer_id,
        orders_stream_watermark.order_status,
        items_stream_watermark.order_item_product_id,
        items_stream_watermark.order_item_quantity,
        items_stream_watermark.order_item_subtotal
    )
)

In [19]:
ventas.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_item_product_id: integer (nullable = true)
 |-- order_item_quantity: integer (nullable = true)
 |-- order_item_subtotal: double (nullable = true)



In [20]:
query_join = (
    ventas
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "truncate",
        False
    )
    .start()
)

26/08/07 20:58:12 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-843616cc-af07-4800-b4ff-baca73fca5b1. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/07 20:58:12 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
[Stage 7:=====>                                                  (20 + 2) / 200]

In [22]:
query_join.stop()

In [20]:
join_query = (
    ventas
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .trigger(
        processingTime="5 seconds"
    )
    .start()
)

26/08/12 01:25:16 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c747b346-40e2-4946-8d67-7fd174de2c3b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/12 01:25:16 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+-----------------+------------+---------------------+-------------------+-------------------+
|order_id|order_customer_id|order_status|order_item_product_id|order_item_quantity|order_item_subtotal|
+--------+-----------------+------------+---------------------+-------------------+-------------------+
+--------+-----------------+------------+---------------------+-------------------+-------------------+



26/08/12 01:25:55 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 39479 milliseconds
                                                                                

-------------------------------------------
Batch: 1
-------------------------------------------
+--------+-----------------+------------+---------------------+-------------------+-------------------+
|order_id|order_customer_id|order_status|order_item_product_id|order_item_quantity|order_item_subtotal|
+--------+-----------------+------------+---------------------+-------------------+-------------------+
|68954   |5                |CLOSED      |1292                 |2                  |119.94             |
|68953   |4234             |PENDING     |276                  |2                  |63.98              |
|68953   |4234             |PENDING     |769                  |3                  |44.97              |
|68953   |4234             |PENDING     |406                  |4                  |396.0              |
|68950   |509              |COMPLETE    |709                  |4                  |399.96             |
|68950   |509              |COMPLETE    |734                  |1       

26/08/12 01:26:28 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 33374 milliseconds
                                                                                

-------------------------------------------
Batch: 2
-------------------------------------------
+--------+-----------------+------------+---------------------+-------------------+-------------------+
|order_id|order_customer_id|order_status|order_item_product_id|order_item_quantity|order_item_subtotal|
+--------+-----------------+------------+---------------------+-------------------+-------------------+
|68956   |517              |PENDING     |612                  |3                  |509.97             |
|68956   |517              |PENDING     |334                  |1                  |38.0               |
|68959   |9068             |PENDING     |808                  |3                  |59.97              |
|68959   |9068             |PENDING     |1287                 |2                  |119.98             |
|68957   |6295             |PROCESSING  |731                  |4                  |252.0              |
|68957   |6295             |PROCESSING  |1275                 |3       

26/08/12 01:26:59 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 30207 milliseconds
26/08/12 01:27:29 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 30008 milliseconds


-------------------------------------------
Batch: 3
-------------------------------------------
+--------+-----------------+------------+---------------------+-------------------+-------------------+
|order_id|order_customer_id|order_status|order_item_product_id|order_item_quantity|order_item_subtotal|
+--------+-----------------+------------+---------------------+-------------------+-------------------+
|68962   |9915             |PENDING     |1159                 |2                  |130.0              |
|68962   |9915             |PENDING     |361                  |1                  |129.99             |
|68962   |9915             |PENDING     |161                  |4                  |179.96             |
|68962   |9915             |PENDING     |208                  |4                  |7999.96            |
|68962   |9915             |PENDING     |1071                 |3                  |689.97             |
|68963   |2987             |PROCESSING  |1323                 |4       

[Stage 34:=========================================>            (152 + 2) / 200]

In [22]:
join_query.stop()

# Practica 4
### Materializar el proceso stream en un formato de archivo 

In [25]:
ventas = (
    orders_stream_watermark
    .join(
        items_stream_watermark,
        orders_stream_watermark.order_id ==
        items_stream_watermark.order_item_order_id,
        "inner"
    )
)

streaming_query = (
    ventas
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option(
        "path",
        "hdfs://namenode:8020/lambda/speed"
    )
    .option(
        "checkpointLocation",
        "hdfs://namenode:8020/lambda/checkpoint_speed"
    )
    .trigger(
        processingTime="2 minutes"
    )
    .start()
)

26/08/12 01:49:56 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
[Stage 37:===============>                                       (56 + 2) / 200]

# ENTREGABLE PARA EL MODULO 

### 1 Trabajar el modelo lambda en las capas:
#### Capa Bacth
      a ) Con el notebook "Batch Propuesto.ipynb" hacer correr, con los ajustes si corresponde, un proceso batchero de cada 10 minutos en la capa LAMBDA/RAW/BATCH
          Puede ser > Cada 20 miutos toda la tabla 
          Puede ser > Carga inical total, y cada 20 minutos solo data incremental 
      b ) Con el notebook "Batch Propuesto.ipynb" con los ajustes un proceso JOIN de cada 20 minutos en la capa LAMBDA/CLEANSED/BATCH
#### Capa Stream 
      a ) Con el notebook "5 Streaming.ipynb" hacer correr, procesamiento de JOIN streamin cada 2 minutos en ca capa SPEED

#### Opcional Capa Serving
      a ) Contruir notebook para el servicio que sumariza la data de los ultimos 20 minutos con la data de los dos minutos ultimos.